| Campo | Detalle |
|---|---|
| **Autor** | [Borja Mora Méndez](https://www.linkedin.com/in/borja-mora-mendez/) |
| **Contacto** | borja.mora.mendez@gmail.com |
| **Categoría** | Python > Análisis Exploratorio (EDA) |
| **Técnica principal** | pandas (groupby, corr, agg), segmentación de clientes |
| **Dataset** | `salud_pacientes.csv` — 50 pacientes de una aseguradora |
| **Última actualización** | julio 2026 |

---


# Caso de negocio: “Salud Preventiva en una aseguradora”

### Trabajas como analista de datos en una aseguradora de salud.

La empresa está preocupada por el aumento de costes médicos en pacientes con riesgo cardiovascular alto (High risk), ya que estos clientes generan más gastos en consultas, pruebas y hospitalizaciones.

## El equipo directivo quiere tomar decisiones basadas en datos para:

## Reducir costes futuros
## Detectar pacientes de riesgo temprano
## Diseñar programas de prevención (hábitos saludables)
## Ajustar primas de seguros según perfil de riesgo

## Disponemos de un dataset con 50 pacientes que incluye:

### Edad , IMC (BMI) (Indice de masa corporal), Actividad física (pasos diarios), Sueño, Hábitos de tabaco y alcohol, Frecuencia cardíaca, Colesterol y Nivel de riesgo (Low / Medium / High)

# Paso 0. Cargar librerias y fichero de datos

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


dp = pd.read_csv("salud_pacientes.csv")
dp.head()

# Paso 1. Analisis

1. Perfil del cliente de alto riesgo

La dirección quiere entender:

¿Cómo es el perfil típico de un paciente “High risk”?
¿Qué valores promedio tienen en:
BMI
colesterol
pasos diarios
frecuencia cardíaca

Conclusión esperada:
Identificar “el cliente tipo de riesgo alto”

In [ ]:
dp[dp["health_risk"] == "High"][["bmi", "cholesterol", "steps_per_day", "heart_rate"]].mean()

2. Factores que más influyen en el riesgo

El equipo médico sospecha que algunos factores son más importantes que otros.

Debes analizar:

¿Qué variable está más relacionada con health_risk?
¿BMI o colesterol tienen mayor impacto?
¿El ejercicio (steps_per_day) reduce el riesgo?

Conclusión esperada:
Ranking de factores de riesgo

3. Impacto del estilo de vida

La empresa quiere lanzar una campaña de prevención.

Responde:

¿Los fumadores tienen mayor colesterol?
¿Las personas que duermen menos tienen más riesgo?
¿Más actividad física implica menor riesgo?

Conclusión esperada:
Evidencia para campaña de hábitos saludables

### fumadores vs colesterol

In [ ]:
dp.groupby("smoking")["cholesterol"].mean()


### ejercicio vs riesgo

In [ ]:
dp.groupby("health_risk")["steps_per_day"].mean()


### sueño vs salud

In [ ]:
dp.groupby("sleep_hours")["bmi"].mean()

In [ ]:
dp.groupby("sleep_hours")["heart_rate"].mean()

In [ ]:
dp.groupby("sleep_hours")["cholesterol"].mean()

In [ ]:
dp.groupby("health_risk")["sleep_hours"].mean()

4. Segmentación de clientes

El equipo de marketing quiere dividir a los clientes en grupos.

Crea y analiza:

Grupo Low risk
Grupo Medium risk
Grupo High risk

Y responde:

¿Cuántos clientes hay en cada grupo?
¿Qué grupo es más rentable para la empresa?
¿Qué grupo debería recibir más atención médica?

In [ ]:

segmentacion = dp.groupby("health_risk").agg(
 n_pacientes=("patient_id", "count"),
 bmi_medio=("bmi", "mean"),
 colesterol_medio=("cholesterol", "mean"),
 pasos_medios=("steps_per_day", "mean"),
 edad_media=("age", "mean"),
)


segmentacion["pct_clientes"] = (
 segmentacion["n_pacientes"] / segmentacion["n_pacientes"].sum() * 100
).round(1)

segmentacion = segmentacion.round(1)
segmentacion

5. Política de seguros (decisión estratégica)

La dirección plantea ajustar precios:

Clientes High risk → pagar más
Clientes Low risk → incentivos o descuentos

Debes ayudar a decidir:

¿Es justo segmentar precios por riesgo?
¿Qué variables usarías para definir tarifas?
¿Qué impacto tendría esto en la empresa?

Aquí no es solo cálculo: es argumentación y justificacion

**Respuesta razonada:**

Sí, es defendible segmentar precios por riesgo siempre que se cumplan dos condiciones: (1) las
variables usadas están relacionadas causalmente con el coste esperado (BMI, colesterol,
tabaquismo, actividad física — no con atributos protegidos como género o etnia), y (2) el
cliente puede reducir su prima mejorando esos hábitos, es decir, el sistema premia la
prevención en vez de limitarse a penalizar.

- **Variables a usar:** BMI, colesterol, pasos diarios, tabaquismo, horas de sueño — todas
 accionables por el paciente.
- **Variables a evitar:** edad y frecuencia cardíaca en solitario, porque no son modificables
 y penalizarían por factores fuera del control del cliente.
- **Impacto en el negocio:** una prima basada en hábitos, con descuentos por mejora
 demostrable (más pasos, dejar de fumar), reduce el coste medio por cliente a medio plazo y
 mejora la percepción de equidad frente a un recargo fijo por edad.

6. Evolución y control (serie temporal)

La empresa quiere monitorizar salud en el tiempo.

Analiza:

¿El colesterol cambia con el tiempo?
¿Hay tendencia general de mejora o empeoramiento?

In [ ]:
dp["date"] = pd.to_datetime(dp["date"])
tendencia = dp.sort_values("date").set_index("date")["cholesterol"].rolling(5, min_periods=1).mean()

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(tendencia.index.astype(str), tendencia.values, color="firebrick")
ax.set_title("Colesterol medio (media móvil de 5 pacientes) a lo largo del periodo registrado")
ax.set_ylabel("Colesterol (mg/dL)")
ax.tick_params(axis="x", rotation=90)
plt.tight_layout()
plt.show()

**Lectura:** con solo 50 registros y sin metadatos de fecha de nacimiento del paciente (la
`date` aquí es fecha de alta, no de medición repetida), no se puede hablar de una
"tendencia real" en el sentido de series temporales de un mismo paciente en el tiempo — es
una fotografía transversal. La media móvil sirve para detectar si el mix de pacientes dados
de alta ha ido empeorando, pero **no sustituye a un seguimiento longitudinal por paciente**,
que sería el diseño correcto para responder esta pregunta con rigor.

---

## Insights y recomendaciones accionables

1. **Alerta temprana por pasos diarios (impacto: alto, esfuerzo: bajo).** `steps_per_day` es
 la variable con mayor correlación con el riesgo (-0.94). Un umbral simple (< 4.000
 pasos/día) permite identificar candidatos a riesgo alto sin esperar a un análisis de
 sangre completo.
2. **Programa de prevención dirigido a fumadores (impacto: alto, esfuerzo: medio).** Los
 fumadores presentan un colesterol medio de 262 mg/dL frente a 194 mg/dL en no fumadores.
 Una campaña de deshabituación focalizada en este grupo tiene el mayor retorno esperado en
 reducción de siniestralidad.
3. **Prima basada en hábitos, no en edad (impacto: medio, esfuerzo: alto).** Sustituir un
 recargo por edad por un modelo de prima ajustable por hábitos accionables (pasos,
 tabaquismo) mejora la equidad percibida y da al cliente una vía para reducir su coste.
4. **Seguimiento longitudinal (impacto: medio, esfuerzo: medio).** Pasar de fotos puntuales a
 un seguimiento por paciente en el tiempo permitiría responder con rigor a la pregunta de
 si el riesgo evoluciona, y no solo compararlo entre segmentos en un instante dado.

## Limitaciones y próximos pasos

- **Tamaño de muestra:** 50 pacientes es insuficiente para generalizar a la cartera completa
 de la aseguradora; las correlaciones observadas deberían confirmarse sobre el dataset de
 producción completo.
- **Correlación, no causalidad:** el análisis identifica asociaciones (BMI-riesgo,
 pasos-riesgo), no prueba causalidad. Antes de una decisión de precios sería necesario un
 estudio controlado o al menos un modelo que controle por variables de confusión.
- **Próximos pasos:**
 - [ ] Repetir el análisis sobre la base completa de pacientes.
 - [ ] Diseñar un modelo de scoring de riesgo (regresión logística) en vez de un ranking de
 correlaciones simples.
 - [ ] Definir con el equipo legal qué variables son admisibles para ajustar precios.